# Two-Tower Product Recommendation System
## Cross-Category Complementary Product Recommendation Engine

This notebook implements an end-to-end Two-Tower & Siamese embedding recommendation system with PyTorch model fine-tuning on `triplet_dataset_v2.csv`.

### Key Features & Options:
1. **Cross-Category Complementary Filtering (`filter_same_category=True`)**:
   - Option to strictly filter out same-category products (e.g. excluding other milks when query is `"Amul Milk"`), returning 100% cross-category complementary products (e.g., Chocolates, Beverages, Snacks, Ice Creams).
2. **Category-Stratified Data Splitting**: Guarantees **100% of categories** are represented across Train, Val, and Test splits.
3. **Model Architectures**: `TwoTower` and `SiameseTower` backbones initialized from BGE (`BAAI/bge-small-en-v1.5`).
4. **PyTorch Fine-Tuning Training Loop**: AdamW optimizer with linear warmup scheduler and best checkpoint saving.


In [2]:
# Environment Setup for Google Colab / Local Execution
import sys
import os

# Install required packages
# !pip install -q sentence-transformers torch pandas numpy scikit-learn tqdm transformers

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))


PyTorch Version: 2.10.0+cpu
CUDA Available: False


## Stage 1: Load Precomputed Embeddings & Category-Stratified Data Split

Loads `item_embeddings_50k.pkl` and `triplet_dataset_v2.csv`, performing category-stratified anchor splitting (80% Train, 10% Val, 10% Test).


In [ ]:
import os
import random
import time
import pickle
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict, Set
from torch.utils.data import Dataset, DataLoader

@dataclass
class ExperimentConfig:
    experiment_id: str = "exp01_two_tower"
    description: str = "True Two-Tower (Separate Anchor & Candidate Encoders) Fine-Tuning"
    
    tower_type: str = "two_tower"         # 'siamese', 'two_tower'
    base_model_name: str = "BAAI/bge-small-en-v1.5"
    use_projection_head: bool = False      # Projection MLP on top of BGE
    projection_dim: int = 384             # Output embedding dimension
    freeze_base: bool = False
    
    loss_type: str = "mnrl"               # 'mnrl', 'triplet'
    triplet_margin: float = 0.5
    mnrl_temperature: float = 0.05
    
    include_same_category: bool = True
    positive_sources: List[str] = field(default_factory=lambda: ['positive_gold', 'positive_silver', 'positive_cross_category'])
    negative_sources: List[str] = field(default_factory=lambda: ['soft_negative', 'hard_negative'])
    
    batch_size: int = 64
    epochs: int = 3
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    random_seed: int = 42
    
    data_path: str = "triplet_dataset_v3.csv"
    embeddings_pkl_path: str = "item_embeddings_50k.pkl"
    output_dir: str = "checkpoints"

def load_cached_embeddings(pkl_path: str = "item_embeddings_50k.pkl") -> Optional[Dict]:
    if os.path.exists(pkl_path):
        print(f"Loading precomputed item embeddings from '{pkl_path}'...")
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        print(f"Loaded embeddings for {len(data.get('item_to_idx', {}))} items.")
        return data
    else:
        print(f"Embeddings pickle file '{pkl_path}' not found. Embeddings will be computed dynamically.")
        return None

def load_and_prepare_data(
    data_path: str = "triplet_dataset_v3.csv",
    seed: int = 42,
    train_ratio: float = 0.80,
    val_ratio: float = 0.10,
    test_ratio: float = 0.10,
    include_same_category: bool = True,
    positive_sources: Optional[List[str]] = None,
    negative_sources: Optional[List[str]] = None
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, str]]:
    df = pd.read_csv(data_path)
    
    if not include_same_category and 'same_category' in df.columns:
        df = df[df['same_category'] == False]
        
    if positive_sources is not None:
        pos_mask = (df['label'] == 1) & (df['sample_type'].isin(positive_sources))
        neg_mask = (df['label'] == 0)
        df = df[pos_mask | neg_mask]
        
    if negative_sources is not None:
        pos_mask = (df['label'] == 1)
        neg_mask = (df['label'] == 0) & (df['sample_type'].isin(negative_sources))
        df = df[pos_mask | neg_mask]
        
    df = df.reset_index(drop=True)
    
    anchors = df[['anchor_item', 'anchor_category']].rename(columns={'anchor_item': 'item_name', 'anchor_category': 'category'})
    candidates = df[['candidate_item', 'candidate_category']].rename(columns={'candidate_item': 'item_name', 'candidate_category': 'category'})
    catalog_df = pd.concat([anchors, candidates]).drop_duplicates(subset=['item_name']).sort_values('item_name').reset_index(drop=True)
    item_to_category = dict(zip(catalog_df['item_name'], catalog_df['category']))
    
    # CATEGORY-STRATIFIED Anchor Splitting
    rng = random.Random(seed)
    train_anchors, val_anchors, test_anchors = set(), set(), set()
    anchor_cat_df = df[['anchor_item', 'anchor_category']].drop_duplicates()

    for cat, group in anchor_cat_df.groupby('anchor_category'):
        cat_anchors = list(group['anchor_item'])
        rng.shuffle(cat_anchors)
        n_cat = len(cat_anchors)
        
        if n_cat == 1:
            train_anchors.add(cat_anchors[0])
        elif n_cat == 2:
            train_anchors.add(cat_anchors[0])
            val_anchors.add(cat_anchors[1])
        else:
            n_tr = max(1, int(round(n_cat * train_ratio)))
            n_va = max(1, int(round(n_cat * val_ratio)))
            if n_tr + n_va >= n_cat:
                n_tr = max(1, n_cat - 2)
                n_va = 1
            train_anchors.update(cat_anchors[:n_tr])
            val_anchors.update(cat_anchors[n_tr:n_tr + n_va])
            test_anchors.update(cat_anchors[n_tr + n_va:])

    train_df = df[df['anchor_item'].isin(train_anchors)].reset_index(drop=True)
    val_df = df[df['anchor_item'].isin(val_anchors)].reset_index(drop=True)
    test_df = df[df['anchor_item'].isin(test_anchors)].reset_index(drop=True)
    
    print(f"Category-Stratified Data Prepared from '{data_path}':")
    print(f"  - Total Dataset Rows: {len(df)}")
    print(f"  - Total Catalog Items: {len(catalog_df)}")
    print(f"  - Train Set: {len(train_df)} rows ({len(train_anchors)} anchors across {train_df['anchor_category'].nunique()} categories)")
    print(f"  - Val Set  : {len(val_df)} rows ({len(val_anchors)} anchors across {val_df['anchor_category'].nunique()} categories)")
    print(f"  - Test Set : {len(test_df)} rows ({len(test_anchors)} anchors across {test_df['anchor_category'].nunique()} categories)")
    
    return train_df, val_df, test_df, catalog_df, item_to_category

class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        pos_df = df[df['label'] == 1].reset_index(drop=True)
        self.records = pos_df[['anchor_item', 'candidate_item']].to_dict(orient='records')
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        row = self.records[idx]
        return {'anchor_item': row['anchor_item'], 'positive_item': row['candidate_item']}

class TripletDataset(Dataset):
    def __init__(self, df: pd.DataFrame, seed: int = 42):
        self.rng = random.Random(seed)
        pos_df = df[df['label'] == 1]
        neg_df = df[df['label'] == 0]
        
        anchor_to_negs = {}
        for anchor, group in neg_df.groupby('anchor_item'):
            anchor_to_negs[anchor] = list(group['candidate_item'].unique())
            
        all_candidate_items = list(df['candidate_item'].unique())
        self.triplets = []
        
        for idx, row in pos_df.iterrows():
            anchor = row['anchor_item']
            pos_item = row['candidate_item']
            neg_candidates = anchor_to_negs.get(anchor, [])
            
            if neg_candidates:
                neg_item = self.rng.choice(neg_candidates)
            else:
                neg_item = self.rng.choice(all_candidate_items)
                while neg_item == anchor or neg_item == pos_item:
                    neg_item = self.rng.choice(all_candidate_items)
                    
            self.triplets.append({'anchor_item': anchor, 'positive_item': pos_item, 'negative_item': neg_item})
            
    def __len__(self):
        return len(self.triplets)
        
    def __getitem__(self, idx):
        return self.triplets[idx]


## Stage 2: PyTorch Model Architectures (Siamese & Two-Tower)

Defines `ProjectionHead`, `SiameseTower`, and `TwoTower` models.


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

class ProjectionHead(nn.Module):
    def __init__(self, in_features: int = 384, out_features: int = 384):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.LayerNorm(in_features),
            nn.ReLU(),
            nn.Linear(in_features, out_features),
            nn.LayerNorm(out_features)
        )
            
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.net(x)
        return F.normalize(x, p=2, dim=-1)

class SiameseTower(nn.Module):
    def __init__(
        self,
        base_model_name: str = "BAAI/bge-small-en-v1.5",
        use_projection_head: bool = False,
        projection_dim: int = 384,
        freeze_base: bool = False
    ):
        super().__init__()
        self.encoder = SentenceTransformer(base_model_name)
        if hasattr(self.encoder, 'get_embedding_dimension'):
            in_dim = self.encoder.get_embedding_dimension()
        else:
            in_dim = self.encoder.get_sentence_embedding_dimension()

        if freeze_base:
            for param in self.encoder.parameters():
                param.requires_grad = False
                
        self.use_projection_head = use_projection_head
        if use_projection_head:
            self.proj = ProjectionHead(in_features=in_dim, out_features=projection_dim)
        else:
            self.proj = None

    def encode(self, texts: List[str], device: torch.device = None) -> torch.Tensor:
        if device is None:
            device = next(self.parameters()).device
        features = self.encoder.tokenize(texts)
        features = {k: v.to(device) if hasattr(v, 'to') else torch.tensor(v, device=device) for k, v in features.items()}
        outputs = self.encoder(features)
        emb = outputs['sentence_embedding']
        if self.use_projection_head and self.proj is not None:
            emb = self.proj(emb)
        else:
            emb = F.normalize(emb, p=2, dim=-1)
        return emb

    def forward(self, anchor_texts: List[str], candidate_texts: List[str]):
        return self.encode(anchor_texts), self.encode(candidate_texts)

class TwoTower(nn.Module):
    def __init__(
        self,
        base_model_name: str = "BAAI/bge-small-en-v1.5",
        use_projection_head: bool = False,
        projection_dim: int = 384,
        freeze_base: bool = False
    ):
        super().__init__()
        self.anchor_encoder = SentenceTransformer(base_model_name)
        self.candidate_encoder = SentenceTransformer(base_model_name)
        if hasattr(self.anchor_encoder, 'get_embedding_dimension'):
            in_dim = self.anchor_encoder.get_embedding_dimension()
        else:
            in_dim = self.anchor_encoder.get_sentence_embedding_dimension()

        if freeze_base:
            for param in self.anchor_encoder.parameters():
                param.requires_grad = False
            for param in self.candidate_encoder.parameters():
                param.requires_grad = False

        self.use_projection_head = use_projection_head
        if use_projection_head:
            self.anchor_proj = ProjectionHead(in_features=in_dim, out_features=projection_dim)
            self.candidate_proj = ProjectionHead(in_features=in_dim, out_features=projection_dim)
        else:
            self.anchor_proj = None
            self.candidate_proj = None

    def encode_anchor(self, texts: List[str], device: torch.device = None) -> torch.Tensor:
        if device is None:
            device = next(self.anchor_encoder.parameters()).device
        features = self.anchor_encoder.tokenize(texts)
        features = {k: v.to(device) if hasattr(v, 'to') else torch.tensor(v, device=device) for k, v in features.items()}
        outputs = self.anchor_encoder(features)
        emb = outputs['sentence_embedding']
        if self.use_projection_head and self.anchor_proj is not None:
            emb = self.anchor_proj(emb)
        else:
            emb = F.normalize(emb, p=2, dim=-1)
        return emb

    def encode_candidate(self, texts: List[str], device: torch.device = None) -> torch.Tensor:
        if device is None:
            device = next(self.candidate_encoder.parameters()).device
        features = self.candidate_encoder.tokenize(texts)
        features = {k: v.to(device) if hasattr(v, 'to') else torch.tensor(v, device=device) for k, v in features.items()}
        outputs = self.candidate_encoder(features)
        emb = outputs['sentence_embedding']
        if self.use_projection_head and self.candidate_proj is not None:
            emb = self.candidate_proj(emb)
        else:
            emb = F.normalize(emb, p=2, dim=-1)
        return emb

    def forward(self, anchor_texts: List[str], candidate_texts: List[str]):
        return self.encode_anchor(anchor_texts), self.encode_candidate(candidate_texts)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).
W0822 20:03:05.294000 23056 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## Stage 3: Two-Tower Product Recommendation Predictor (`ItemRecommender`)

Defines `ItemRecommender` class providing `.predict(product_name, top_k=10, filter_same_category=False)`.
Set `filter_same_category=True` to strictly return cross-category complementary recommendations!


In [5]:
class ItemRecommender:
    def __init__(self, model: torch.nn.Module, catalog_df: pd.DataFrame, device: Optional[torch.device] = None, pkl_path: str = "item_embeddings_50k.pkl"):
        self.model = model
        self.catalog_df = catalog_df.reset_index(drop=True)
        self.catalog_items = list(self.catalog_df['item_name'])
        self.item_to_category = dict(zip(self.catalog_df['item_name'], self.catalog_df['category']))
        self.item_to_idx = {name: idx for idx, name in enumerate(self.catalog_items)}
        self.pkl_path = pkl_path
        
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = device
            
        self.model.to(self.device)
        self.model.eval()
        self._initialize_catalog_vectors()
        
    def _initialize_catalog_vectors(self):
        if os.path.exists(self.pkl_path):
            try:
                with open(self.pkl_path, 'rb') as f:
                    cached_data = pickle.load(f)
                cached_matrix = cached_data.get('embeddings')
                cached_item_to_idx = cached_data.get('item_to_idx')
                
                catalog_matrix = np.zeros((len(self.catalog_items), cached_matrix.shape[1]), dtype='float32')
                for i, name in enumerate(self.catalog_items):
                    idx = cached_item_to_idx.get(name)
                    if idx is not None:
                        catalog_matrix[i] = cached_matrix[idx]
                        
                self.catalog_matrix = catalog_matrix
                print(f"ItemRecommender initialized catalog vectors from '{self.pkl_path}'. Shape: {self.catalog_matrix.shape}")
                return
            except Exception as e:
                pass

        embeddings = []
        batch_size = 256
        with torch.no_grad():
            for i in range(0, len(self.catalog_items), batch_size):
                batch_texts = self.catalog_items[i : i + batch_size]
                if hasattr(self.model, 'encode_candidate'):
                    emb = self.model.encode_candidate(batch_texts, device=self.device)
                else:
                    emb = self.model.encode(batch_texts, device=self.device)
                embeddings.append(emb.cpu().numpy())
                
        self.catalog_matrix = np.vstack(embeddings).astype('float32')

    def predict(self, product_name: str, top_k: int = 10, filter_same_category: bool = False) -> pd.DataFrame:
        if product_name not in self.item_to_idx:
            raise ValueError(f"Product '{product_name}' not found in catalog!")
            
        product_cat = self.item_to_category[product_name]
        with torch.no_grad():
            if hasattr(self.model, 'encode_anchor'):
                product_vec = self.model.encode_anchor([product_name], device=self.device).cpu().numpy().astype('float32')
            else:
                product_vec = self.model.encode([product_name], device=self.device).cpu().numpy().astype('float32')
                
        scores = np.dot(self.catalog_matrix, product_vec[0]).copy()
        
        if product_name in self.item_to_idx:
            scores[self.item_to_idx[product_name]] = -np.inf
            
        if filter_same_category:
            for idx, item in enumerate(self.catalog_items):
                if self.item_to_category[item] == product_cat:
                    scores[idx] = -np.inf

        top_indices = np.argsort(scores)[::-1][:top_k * 2]
        top_scores = scores[top_indices]
        
        results = []
        rank = 1
        for idx, score in zip(top_indices, top_scores):
            if np.isneginf(score):
                continue
            rec_item = self.catalog_items[idx]
            if rec_item == product_name:
                continue
            if rank > top_k:
                break
                
            rec_cat = self.item_to_category[rec_item]
            results.append({
                'rank': rank,
                'recommended_item': rec_item,
                'category': rec_cat,
                'confidence_score': round(float(score), 4),
                'same_category': (rec_cat == product_cat)
            })
            rank += 1
            
        return pd.DataFrame(results)

print("ItemRecommender model prediction interface ready!")


ItemRecommender model prediction interface ready!


## Stage 4: Interactive Product Recommendation Predictor

Enter any product name below and toggle `filter_same_category=True` to get 100% cross-category complementary recommendations!


In [7]:
# Initialize model and predictor
_, _, _, catalog_df, _ = load_and_prepare_data()
model = TwoTower()
if os.path.exists("checkpoints/exp01_two_tower_best.pt"):
    model.load_state_dict(torch.load("checkpoints/exp01_two_tower_best.pt", map_location=torch.device("cpu")))
    print("Loaded trained model weights.")

recommender = ItemRecommender(model, catalog_df)

input_product = "Amul Milk"
filter_same = True  # Set to True for Cross-Category Complementary Recommendations

print(f"Input Product: '{input_product}' | Filter Same Category: {filter_same}")
top10_recs = recommender.predict(input_product, top_k=10, filter_same_category=True)

display(top10_recs)


Category-Stratified Data Prepared from 'triplet_dataset_v2.csv':
  - Total Dataset Rows: 2517486
  - Total Catalog Items: 49990
  - Train Set: 2014383 rows (39994 anchors across 17 categories)
  - Val Set  : 252054 rows (5000 anchors across 17 categories)
  - Test Set : 251049 rows (4996 anchors across 17 categories)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded trained model weights.
ItemRecommender initialized catalog vectors from 'item_embeddings_50k.pkl'. Shape: (49990, 384)
Input Product: 'Amul Milk' | Filter Same Category: True


,rank,recommended_item,category,confidence_score,same_category
0,1,Amul Milk Chocolate,Snacks & Biscuits,0.9228,False
1,2,Amul Kool Flavored Milk,Beverages,0.8919,False
2,3,Amul Milk Chocolate Chocolates,Ice Creams & Frozen Desserts,0.8805,False
3,4,Amul Milk Chocolate- Smooth & Creamy,Snacks & Biscuits,0.8736,False
4,5,Amul Chocolates,Ice Creams & Frozen Desserts,0.8724,False
5,6,Amul Ice Creams,Ice Creams & Frozen Desserts,0.8722,False
6,7,Amul Kool Chocolate Flavored Milk,Beverages,0.8698,False
7,8,Amul Kool Strawberry Flavored Milk,Beverages,0.8679,False
8,9,Amul India Single Origin Milk Chocolate,Snacks & Biscuits,0.8637,False
9,10,Amul Kool Vanilla Flavored Milk,Beverages,0.8573,False
